# Laboratori 11: Bredhjet e rastit, difuzioni dhe zinxhirët Markov

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si lind difuzioni nga hapa të rastit dhe si përshkruhet kujtesa me një matricë kalimi?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- nxjerrë mesataren dhe variancën e bredhjes simetrike;
- lidhë bredhjen me koeficientin e difuzionit;
- simulojë kohën e parë të kalimit;
- llogarisë një shpërndarje stacionare;
- kontrollojë balancën e detajuar;
- nxjerrë kompleksitetin $\Theta(MN)$ të një ansambli;
- vlerësojë autokorrelacionin dhe madhësinë efektive të mostrës.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Ky laborator lidh një model mikroskopik diskret me një ekuacion makroskopik të vazhdueshëm. Hapat individualë janë të rastit, por mesatarja dhe varianca e ansamblit ndjekin ligje të sakta. Duhet dalluar një trajektore e vetme nga një ansambël trajektoresh dhe mesatarja kohore nga mesatarja e ansamblit.

Në pjesën Markov, matrica e kalimit është modeli. Simulimi prodhon një realizim të këtij modeli, ndërsa shumëzimi me matricën përhap të gjithë shpërndarjen. Të dyja duhet të përputhen statistikisht, por nuk janë e njëjta llogaritje.


## 1. Bredhja simetrike në një përmasë

Në çdo hap, $\xi_i=+\ell$ ose $-\ell$ me probabilitet $1/2$. Pas $N$ hapash,

$$X_N=\sum_{i=1}^{N}\xi_i.$$

Meqë $\mathbb E[\xi_i]=0$ dhe $\operatorname{Var}(\xi_i)=\ell^2$,

$$\mathbb E[X_N]=0,
\qquad \operatorname{Var}(X_N)=N\ell^2.$$

Zhvendosja tipike është $\sqrt{\mathbb E[X_N^2]}=\ell\sqrt N$, jo $N\ell$. Rritja me rrënjë katrore është shenja e difuzionit.


## 2. Kufiri difuziv

Nëse çdo hap zgjat $\Delta t$, atëherë $t=N\Delta t$. Ekuacioni i difuzionit në një përmasë parashikon

$$\langle X^2(t)\rangle=2Dt.$$

Krahasimi me $N\ell^2$ jep

$$D=\frac{\ell^2}{2\Delta t}.$$

Për të marrë një kufi të vazhdueshëm të fundëm, $\ell$ dhe $\Delta t$ zvogëlohen duke mbajtur raportin $\ell^2/(2\Delta t)$ konstant.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
def bredhje_1d(M,N,hap=1.0,rng=rng):
    hapa=hap*np.where(rng.random((M,N))<.5,-1.0,1.0)
    return np.column_stack([np.zeros(M),np.cumsum(hapa,axis=1)])

M,N=5000,1000
X=bredhje_1d(M,N)
print('mesatarja finale=',X[:,-1].mean())
print('MSD finale=',np.mean(X[:,-1]**2),'pritja=',N)


In [ ]:
n=np.arange(N+1); msd=np.mean(X**2,axis=0)
koef=np.polyfit(n[100:],msd[100:],1)
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(n,msd,label='MSD e matur'); ax.plot(n,koef[0]*n+koef[1],'--',label=f'pjerrësia={koef[0]:.3f}')
ax.set(xlabel='numri i hapave',ylabel=r'$\langle X^2\rangle$'); ax.grid(alpha=.3); ax.legend(); plt.show()


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 3.1 Kompleksiteti i ansamblit

Për $M$ realizime dhe $N$ hapa gjenerohen dhe mblidhen $MN$ vlera:

$$T(M,N)=aMN+b\in\Theta(MN).$$

Ruajtja e të gjitha trajektoreve kërkon $\Theta(MN)$ kujtesë. Nëse kërkohet vetëm pozicioni final ose MSD-ja rrjedhëse, kujtesa mund të ulet në $\Theta(M)$. Dy algoritme me të njëjtën kohë asimptotike mund të kenë kërkesa kujtese shumë të ndryshme.


In [ ]:
Ns=np.array([100,200,400,800,1600]); ts=[]
for N in Ns:
    t0=perf_counter(); bredhje_1d(2000,int(N)); ts.append(perf_counter()-t0)
print('pjerrësia ndaj N=',np.polyfit(np.log(Ns),np.log(ts),1)[0])


## 4. Kufijtë dhe koha e parë e kalimit

Me kufij thithës në $\pm L$, koha e parë e kalimit është hapi i parë

$$\tau=\min\{n\geq0:|X_n|\geq L\}.$$

Trajektorja duhet ndalur në goditjen e parë; vazhdimi pas kufirit do të ndryshonte ndryshoren që po matim. Për bredhjen simetrike, koha tipike rritet si $L^2$.


In [ ]:
def kohe_e_pare(L,M=5000,max_hapa=200_000,rng=rng):
    x=np.zeros(M,dtype=int); aktive=np.ones(M,dtype=bool); tau=np.full(M,max_hapa)
    for n in range(1,max_hapa+1):
        idx=np.flatnonzero(aktive)
        if len(idx)==0: break
        x[idx]+=np.where(rng.random(len(idx))<.5,-1,1)
        goditje=idx[np.abs(x[idx])>=L]
        tau[goditje]=n; aktive[goditje]=False
    return tau

for L in [5,10,20,40]:
    tau=kohe_e_pare(L,M=3000,max_hapa=50_000)
    print(f'L={L:2d}: mesatarja tau={tau.mean():.1f}, tau/L²={tau.mean()/L**2:.3f}')


## 5. Zinxhiri Markov dhe matrica e kalimit

Në një zinxhir me gjendje të fundme,

$$P_{ij}=\Pr(X_{n+1}=j\mid X_n=i),$$

dhe çdo rresht plotëson $\sum_jP_{ij}=1$. Nëse shpërndarja rresht është $\mathbf p_n$, atëherë

$$\mathbf p_{n+1}=\mathbf p_nP.$$

Shpërndarja stacionare plotëson

$$\boldsymbol\pi=\boldsymbol\pi P,qquad \sum_i\pi_i=1.$$

Kjo është një problem vektori vetjak me vlerë vetjake $1$.


In [ ]:
P=np.array([[.8,.2,0.],[.1,.7,.2],[0.,.3,.7]])
p=np.array([1.,0.,0.])
for _ in range(50): p=p@P
w,V=np.linalg.eig(P.T); i=np.argmin(np.abs(w-1)); pi=np.real(V[:,i]); pi/=pi.sum()
print('pas 50 hapash:',p,'stacionare:',pi,'kontroll:',pi@P-pi)


## 6. Balanca e detajuar

Një zinxhir është i kthyeshëm ndaj $\pi$ nëse

$$\pi_iP_{ij}=\pi_jP_{ji}$$

për çdo çift. Duke mbledhur mbi $i$ merret stacionariteti. Balanca e detajuar është kusht i mjaftueshëm, jo kusht i domosdoshëm për stacionaritetin.


In [ ]:
rryma=pi[:,None]*P-pi[None,:]*P.T
print('maksimumi i shkeljes së balancës së detajuar=',np.max(np.abs(rryma)))


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 7.1 Ruajtja e probabilitetit dhe gabimet e rrumbullakimit

Përsëritja $p\leftarrow pP$ duhet të ruajë $\sum_ip_i=1$ dhe $p_i\geq0$. Devijime shumë të vogla mund të korrigjohen me normalizim, por vlera negative të konsiderueshme tregojnë matricë të gabuar ose algoritëm të papërshtatshëm. Për kohë shumë të gjata, fuqizimi i drejtpërdrejtë dhe simulimi kanë role të ndryshme: i pari përhap probabilitetin, i dyti prodhon trajektore.


In [ ]:
p=np.array([.2,.3,.5]); shuma=[]
for _ in range(100_000):
    p=p@P
    shuma.append(p.sum())
print('devijimi maksimal i shumës=',np.max(np.abs(np.array(shuma)-1)))


## 8. Autokorrelacioni dhe madhësia efektive

Mostrat e një zinxhiri Markov nuk janë të pavarura. Për autokorrelacionet $\rho_k$, koha e integruar është

$$\tau_{\rm int}=1+2\sum_{k\geq1}\rho_k,$$

dhe madhësia efektive afërsisht

$$N_{\rm eff}=\frac{N}{\tau_{\rm int}}.$$

Prerja e shumës bëhet kur autokorrelacionet bëhen të vogla ose të zhurmshme. Përdorimi i $N$ si numër mostrash të pavarura do ta nënvlerësonte gabimin.


In [ ]:
def simulo_markov(P,N,fillim=0,rng=rng):
    s=np.empty(N,dtype=int); s[0]=fillim
    for n in range(1,N): s[n]=rng.choice(len(P),p=P[s[n-1]])
    return s

def acf(x,max_lag=100):
    x=np.asarray(x,float); x-=x.mean(); v=x@x
    return np.array([x[:len(x)-k]@x[k:]/v for k in range(max_lag+1)])

s=simulo_markov(P,50_000); obs=(s==2).astype(float); rho=acf(obs,100)
poz=rho[1:][rho[1:]>0]; tau=1+2*poz.sum(); Neff=len(obs)/tau
print('N=',len(obs),'tau_int≈',tau,'N_eff≈',Neff)


## 9. Shembulli i plotë dhe punishte dinamike

Paraqitni një grup bredhjesh dhe histogramin final. Ndryshoni $M$, $N$, driftin ose kufirin. Grafikët dinamikë duhet të shoqërohen nga MSD-ja, jo vetëm nga trajektore të bukura.


In [ ]:
def animim_bredhjesh(M=25,N=300,DINAMIK=False):
    X=bredhje_1d(M,N); fig,ax=plt.subplots(figsize=(7,4))
    if DINAMIK:
        for n in range(1,N+1,5):
            ax.clear(); ax.plot(np.arange(n+1),X[:,:n+1].T,alpha=.35)
            ax.set(xlim=(0,N),ylim=(-3*np.sqrt(N),3*np.sqrt(N)),xlabel='n',ylabel='X_n'); plt.pause(.03)
    else:
        ax.plot(X.T,alpha=.35); ax.set(xlabel='n',ylabel='$X_n$'); ax.grid(alpha=.3)
    plt.show(); return X

Xdemo=animim_bredhjesh(DINAMIK=False)


### Bredhja me drift

Nëse $\Pr(\xi=+\ell)=p$ dhe $\Pr(\xi=-\ell)=1-p$, atëherë

$$\mathbb E[\xi]=\ell(2p-1),$$

$$\operatorname{Var}(\xi)=4\ell^2p(1-p).$$

Për $N$ hapa të pavarur,

$$\mathbb E[X_N]=N\ell(2p-1),\qquad
\operatorname{Var}(X_N)=4N\ell^2p(1-p).$$

Drifti rritet si $N$, ndërsa gjerësia si $\sqrt N$.


In [ ]:
for pdrift in [.5,.55,.7]:
    M,N=10000,500
    Xi=np.where(rng.random((M,N))<pdrift,1,-1).sum(axis=1)
    print(pdrift,Xi.mean(),N*(2*pdrift-1),Xi.var(),4*N*pdrift*(1-pdrift))


### Shpejtësia e përzierjes dhe hendeku spektral

Për një zinxhir të fundëm, konvergjenca drejt stacionaritetit kontrollohet shpesh nga moduli i vlerës së dytë vetjake. Hendeku $1-|\lambda_2|$ i vogël nënkupton përzierje të ngadaltë dhe autokorrelacion të gjatë. Kjo lidh analizën spektrale me madhësinë efektive të mostrës.


In [ ]:
eig=np.sort(np.abs(np.linalg.eigvals(P)))[::-1]
print('modulet vetjake=',eig,'hendeku spektral=',1-eig[1])
p0=np.array([1.,0.,0.]); dist=[]; pp=p0.copy()
for n in range(40):
    dist.append(.5*np.sum(np.abs(pp-pi))); pp=pp@P
print('distanca totale pas 0,10,20,39 hapash:',[dist[i] for i in [0,10,20,39]])


### Koha e parë e kalimit ka shpërndarje të gjerë

Mesatarja e $\tau$ nuk përshkruan e vetme eksperimentin. Disa trajektore godasin kufirin shpejt, të tjera qëndrojnë gjatë pranë origjinës. Raportoni medianën, kuantilet dhe pjesën që nuk e ka goditur kufirin brenda `max_hapa`. Censurimi i heshtur i trajektoreve të gjata e zhvendos rezultatin.


In [ ]:
tau=kohe_e_pare(20,M=10000,max_hapa=10000)
print('mesatare=',tau.mean(),'medianë=',np.median(tau),'q90=',np.quantile(tau,.9),'censuruar=',np.mean(tau==10000))


### Pyetje për vetëkontroll

1. Pse zhvendosja mesatare dhe zhvendosja tipike janë të ndryshme?
2. Si ulet kujtesa pa ndryshuar koston $O(MN)$?
3. Çfarë lidhjeje ka hendeku spektral me autokorrelacionin?
4. Pse trajektoret e censuruara duhen raportuar në kohën e parë të kalimit?


# Pjesa II — GNU Octave

## 10. Bredhja dhe MSD-ja

```octave
rand('seed',2026); M=5000; N=1000;
hapa=2*(rand(M,N)>=0.5)-1; X=[zeros(M,1),cumsum(hapa,2)];
msd=mean(X.^2,1); n=0:N;
plot(n,msd,n,n,'k--'); grid on; xlabel('n'); ylabel('MSD');
fprintf('MSD finale %.5f, pritja %d\n',msd(end),N);
```

## 11. Zinxhiri Markov

```octave
P=[.8,.2,0;.1,.7,.2;0,.3,.7]; p=[1,0,0];
for n=1:50, p=p*P; end
[V,D]=eig(P'); [~,i]=min(abs(diag(D)-1)); pi=V(:,i); pi=pi/sum(pi);
disp(p); disp(pi');
rryma=pi.*P-pi'.*P';
fprintf('shkelja DB %.3e\n',max(abs(rryma(:))));
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = 'import numpy as np\nrng=np.random.default_rng(2026)\nfor N in [100,1000,10000]:\n M=5000; X=np.where(rng.random((M,N))<.5,-1,1).sum(axis=1)\n print(N,X.mean(),np.mean(X*X))\n'
script_path = script_dir / 'laboratori_11_bredhjet_markov.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_11_bredhjet_markov.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Nxora $\langle X_N^2\rangle=N\ell^2$.
- [ ] Llogarita koston dhe kujtesën veçmas.
- [ ] Kontrollova stacionaritetin dhe balancën e detajuar.
- [ ] Raportova $N_{eff}$, jo vetëm $N$.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
